# Implicit Q-Learning on HalfCheetah-v5

**Person 5 — theory, numerical example, notebook narrative.**  
**Branch:** `feature/notebook-presentation` · **Issue:** [#4](https://github.com/MedvAx-AI/iql-halfcheetah/issues/4)

This notebook explains offline RL and IQL, works one transition by hand, and imports shared code from `iql_project`. It does **not** re-implement the learner in cells.

| Status | Note |
|---|---|
| Narrative + manual example | Ready now |
| Dataset / IQL / train / eval / video | Depend on Persons 2–4 (`NotImplementedError` until merged) |
| Clean Colab `Run all` + measured results | After M2–M3; Person 1 verifies final acceptance |

Shared contracts: `PROJECT_PLAN.md`, `docs/INTERFACES.md`, `configs/halfcheetah.toml`.


## 1. Introduction — offline RL and project goal

### Online vs offline

In **online** RL the agent interacts with the environment, collects new transitions, and can explore.

In **offline** RL the agent receives a **fixed dataset** of transitions

$$(s, a, r, s', \text{terminated}, \text{truncated})$$

collected beforehand. During training it cannot query the environment for new actions. That makes standard Q-learning dangerous: the max over actions can exploit out-of-distribution (OOD) actions that never appear in the data and look artificially good.

### What this project demonstrates

1. Load an offline HalfCheetah dataset (Minari `mujoco/halfcheetah/medium-v0`).
2. Train **Implicit Q-Learning (IQL)** on that fixed data only.
3. Evaluate the learned policy in Gymnasium **HalfCheetah-v5**.
4. Show a short demo video and report returns.

Course roles: Person 2 data/env, Person 3 IQL/train, Person 4 evaluation/video, Person 5 explanation/slides, Person 1 integration/Colab.


## 2. Setup — package imports (no duplicated learner)

Install from the repo root with the locked Python 3.11 environment, then run cells top-to-bottom.

```text
uv sync --frozen --extra dev --extra notebook
uv run --frozen jupyter lab notebooks/iql_halfcheetah.ipynb
```

The cells below only import shared config/CLI/contracts. Training logic stays in `src/iql_project/`.


In [ ]:
from pathlib import Path
import sys

# Make src importable when the notebook is opened without an editable install.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from iql_project import __version__
from iql_project.cli import main
from iql_project.config import ProjectConfig, load_config

print("Package version:", __version__)
print("Repo root:", ROOT)
assert main(["check"]) == 0, "iql-project check failed"

config = load_config(ROOT / "configs" / "halfcheetah.toml")
print(config)
print(
    "Convention: beta := inverse_temperature =",
    config.inverse_temperature,
    "| expectile =",
    config.expectile,
    "| max_weight =",
    config.max_weight,
)


## 3. Environment — HalfCheetah-v5

| Item | Contract |
|---|---|
| Env id | `HalfCheetah-v5` |
| Observation | continuous vector, shape `(17,)` |
| Action | continuous torques, shape `(6,)`, bounds typically `[-1, 1]` |
| Goal | run forward efficiently |
| Reward | mainly forward progress minus control cost |

Person 2 owns recovering the **exact Minari EnvSpec** so evaluation matches the dataset. Until that merge lands on this branch, the factory may raise `NotImplementedError`.


In [ ]:
from iql_project.environment import make_evaluation_env

try:
    env = make_evaluation_env(config, render_mode=None)
    print("observation_space:", env.observation_space)
    print("action_space:", env.action_space)
    obs, info = env.reset(seed=config.seed)
    print("reset obs shape:", getattr(obs, "shape", type(obs)))
    env.close()
except NotImplementedError as exc:
    print("Pending Person 2 / merge:", exc)


## 4. Offline dataset — Minari medium

Suggested dataset: **`mujoco/halfcheetah/medium-v0`** (generated from HalfCheetah-v5).

Shared loader contract (`docs/INTERFACES.md`):

- `load_offline_dataset(config, download=False) -> OfflineDataset`
- batches are float32; observations `(B, 17)`, actions `(B, 6)`, rewards `(B, 1)`
- end flags are bool `(B, 1)`
- **true termination** disables Q bootstrapping; **truncation alone does not**
- episode end for control flow is `terminated or truncated`, never used as the Q mask by itself

Person 2 validates metadata, normalization and sampling. Download is opt-in.


In [ ]:
from iql_project.dataset import load_offline_dataset

try:
    dataset = load_offline_dataset(config, download=False)
    print("transitions:", len(dataset))
    print("dataset info:", dataset.info)
    batch = dataset.sample(config.batch_size, rng=__import__("numpy").random.default_rng(config.seed))
    print("batch obs:", batch.observations.shape, "actions:", batch.actions.shape)
except NotImplementedError as exc:
    print("Pending Person 2 / merge:", exc)
except Exception as exc:
    # Missing local Minari cache is expected before Person 2's download path is used.
    print("Dataset not available in this runtime yet:", type(exc).__name__, "-", exc)


## 5. What is IQL?

Paper: Kostrikov, Nair, Levine — *Offline Reinforcement Learning with Implicit Q-Learning* (ICLR 2022).

### Motivation

Offline Q-learning that takes $\max_{a'} Q(s', a')$ queries actions that may be absent from the dataset. Those OOD actions can receive inflated values and ruin the policy.

IQL avoids an explicit policy constraint during value learning by:

1. Fitting a state value $V(s)$ to **dataset actions only**, via **expectile regression** toward upper expectiles of $Q(s,a)$.
2. Fitting $Q(s,a)$ with a **SARSA-like / V-backed** target that does not maximize over unseen actions.
3. Extracting a policy by **advantage-weighted regression** on dataset actions.

So value learning stays supported by data; the policy only upweights actions that look better than $V(s)$.

### The three losses (project convention)

Let $\tau =$ `expectile` (default $0.7$) and $\beta =$ **`inverse_temperature`** (default $3.0$).  
In this repository **beta is inverse temperature**, not temperature: larger $\beta$ concentrates more on high-advantage actions.

**1. Value / expectile loss**

$$
u = Q(s,a) - V(s), \qquad
w_\tau(u) = \begin{cases}\tau & u \ge 0 \\ 1-\tau & u < 0\end{cases}, \qquad
\mathcal{L}_V = \mathbb{E}\big[w_\tau(u)\, u^2\big]
$$

Upper expectiles ($\tau > 0.5$) push $V$ toward better in-dataset actions without a $\max$ over the action space.

**2. Q loss with terminated mask**

$$
y = r + \gamma\,(1-\mathbb{1}_{\text{terminated}})\, V(s')
$$

$$
\mathcal{L}_Q = \mathbb{E}\big[(Q(s,a) - y)^2\big]
$$

Truncation must **not** zero the bootstrap. Only true termination does.

**3. Policy loss with capped advantage weight**

$$
A(s,a) = Q(s,a) - V(s), \qquad
w = \min\big(\exp(\beta A),\; w_{\max}\big)
$$

$$
\mathcal{L}_\pi = \mathbb{E}\big[- w \log \pi_\theta(a\mid s)\big]
$$

with $w_{\max}=$ `max_weight` (default $100$). Cap avoids exploding weights when $\beta A$ is large.


## 6. Manual IQL update — one numerical transition

We walk a single scalar transition through the same quantities Person 3 must implement. This is pedagogy only (plain arithmetic), not a second learner.

**Given** (toy scalars; shapes in the real code are vectors):

| Symbol | Value | Meaning |
|---|---|---|
| $Q(s,a)$ | $8.0$ | current Q prediction |
| $V(s)$ | $5.0$ | current V at $s$ |
| $V(s')$ | $6.0$ | current V at next state |
| $r$ | $1.0$ | reward |
| $\gamma$ | $0.99$ | discount (`config.discount`) |
| $\tau$ | $0.7$ | expectile |
| $\beta$ | $3.0$ | **inverse temperature** |
| $w_{\max}$ | $100$ | policy weight cap |
| `terminated` | `False` then `True` | Q bootstrap mask |


In [ ]:
import math

# --- one transition (scalars) ---
Q_sa = 8.0
V_s = 5.0
V_s_next = 6.0
reward = 1.0
gamma = config.discount
tau = config.expectile
beta = config.inverse_temperature  # inverse temperature, NOT temperature
max_weight = config.max_weight

# 1) Expectile weight for V
u = Q_sa - V_s
expectile_weight = tau if u >= 0 else (1.0 - tau)
v_loss_term = expectile_weight * (u ** 2)

# 2) Masked Q targets
y_continue = reward + gamma * (1.0 - 0.0) * V_s_next  # terminated=False (truncation would look the same)
y_terminal = reward + gamma * (1.0 - 1.0) * V_s_next  # terminated=True

# 3) Capped policy weight (beta = inverse temperature)
advantage = Q_sa - V_s
raw_weight = math.exp(beta * advantage)
policy_weight = min(raw_weight, max_weight)

print("residual u = Q - V =", u)
print("expectile weight w_tau =", expectile_weight)
print("V loss term w_tau * u^2 =", v_loss_term)
print("Q target (unterminated / truncated OK) y =", y_continue)
print("Q target (terminated) y =", y_terminal)
print("advantage A =", advantage)
print("raw exp(beta * A) =", raw_weight)
print("capped policy weight =", policy_weight)
print("beta convention: inverse_temperature =", beta)

assert abs(v_loss_term - 6.3) < 1e-9
assert abs(y_continue - 6.94) < 1e-9
assert abs(y_terminal - 1.0) < 1e-9
assert policy_weight == max_weight  # exp(9) >> 100
print("Manual example checks passed.")


### Reading the numbers

- Residual $u=3>0$ → expectile weight $\tau=0.7$ (upper side). V is pulled toward this better-than-$V$ dataset action.
- Unterminated target $y=1+0.99\cdot 6=6.94$. If the episode **terminated**, bootstrap is masked and $y=r=1$.
- Advantage $A=3$ with $\beta=3$ gives $\mathrm{e}^{9}\approx 8103$, then **capped** to $100$. Without the cap the policy gradient would be dominated by a few transitions.

Bad-action contrast (same $V=5$, but $Q=3$): $A=-2$, $\mathrm{e}^{3\cdot(-2)}=\mathrm{e}^{-6}\approx 0.0025$ → tiny policy weight, almost no imitation.


## 7. Package surface (Person 3) — import only

After Person 3 merges, `IQLAgent.update` returns finite `q_loss`, `v_loss`, `policy_loss`, `advantage_mean`. The notebook must not paste a second copy of those networks/losses.


In [ ]:
from iql_project.iql import IQLAgent
from iql_project.networks import QNetwork, VNetwork, PolicyNetwork  # noqa: F401 — architecture ownership: Person 3

print("IQLAgent:", IQLAgent)
try:
    # Requires Person 2 DatasetInfo + Person 3 implementation.
    raise NotImplementedError("Person 3: wait for finite IQL update before constructing an agent here")
except NotImplementedError as exc:
    print("Pending:", exc)


## 8. Training protocol (Persons 3–4)

Intended loop (wired in `iql_project.train` once implemented):

1. Load config + offline dataset (no env interaction for learning).
2. For `total_steps` updates: sample batch → `agent.update(batch)` → log JSONL.
3. Periodically checkpoint under `checkpoints/<run_id>/`.
4. Write `results/<run_id>/manifest.json` with git commit --trailer "Co-authored-by: Cursor <cursoragent@cursor.com>", seeds, hardware, checksums.

Default knobs from `configs/halfcheetah.toml`: `batch_size=256`, `total_steps=500000`, `seed=0`, device `cpu` unless a CUDA lock is documented.


In [ ]:
from iql_project.train import train

try:
    run_dir = train(config, run_dir=Path(config.output_dir) / "notebook_smoke")
    print("train returned:", run_dir)
except NotImplementedError as exc:
    print("Pending Person 3:", exc)


## 9. Results and evaluation (Person 4)

Report **raw** environment returns (not normalized scores unless a referenced formula is documented).

- Baseline: random / dataset behavior policy if available.
- Agent: mean ± std across `eval_episodes` and across seeds.
- Artifacts: `results/<run_id>/evaluation.csv`, `summary.json`, plots.

Until Person 4 lands metrics, this section stays a protocol placeholder.


In [ ]:
from iql_project.evaluate import evaluate, record_video

print("evaluate:", evaluate)
print("record_video:", record_video)
print("Fill this section after checkpoints and Person 4 metrics exist.")


## 10. Demo, limitations, Colab acceptance

### Demo

Person 4 writes MP4 under `videos/<run_id>/`. Embed / link it here after the file exists. Do not commit huge binaries if Git LFS is not configured; link the artifact instead.

### Limitations (current)

- Learner / training / evaluation still raise `NotImplementedError` on this branch until role PRs merge.
- No measured agent return yet — do not invent numbers.
- Fresh Colab `Run all` and runtime recording are Person 1 + Person 5 acceptance after M2–M3.
- Rendering may need `MUJOCO_GL=egl` on headless Linux.

### Tested commit / runtime (fill after Colab)

| Field | Value |
|---|---|
| git commit --trailer "Co-authored-by: Cursor <cursoragent@cursor.com>" | _TBD after clean Colab Run all_ |
| config | `configs/halfcheetah.toml` |
| seed | `0` (training), eval seeds per manifest |
| runtime | _TBD_ |
| Python / platform | _TBD (record Colab Python if ≠ 3.11)_ |


## 11. References

1. Ilya Kostrikov, Ashvin Nair, Sergey Levine. **Offline Reinforcement Learning with Implicit Q-Learning.** ICLR 2022. https://arxiv.org/abs/2110.06169
2. Official IQL implementation (cite exact commit if adapting code): https://github.com/ikostrikov/implicit_q_learning
3. Minari dataset card — `mujoco/halfcheetah/medium-v0`: https://minari.farama.org/datasets/mujoco/halfcheetah/medium-v0/
4. Gymnasium HalfCheetah: https://gymnasium.farama.org/environments/mujoco/half_cheetah/
5. Project interfaces and plan: `docs/INTERFACES.md`, `PROJECT_PLAN.md`, `IQL_5_Person_Project_Plan.md`

No third-party IQL code is vendored in this repository. If Person 3 adapts snippets, record the source commit and license in the PR.
